In [ ]:
%%bash

pip install --upgrade 'scikit-learn<1.6.0'

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.metrics import make_scorer, mean_squared_log_error


import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer


from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score
from sklearn import metrics
from sklearn.model_selection import RandomizedSearchCV



# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the Problem (15 pts)

Answer the following questions.

1. Define the objective: what are you trying to predict?
2. How should you frame this problem as a machine learning problem?
3. How should performance be measured? 


The objective of this project is to **predict the sale price of residential houses** in Ames, Iow, based on various features that describe the property. These features include metrics such as living area, lot size, quality ratings...etc. The main goal is to develop a model that can estimate the sale price of a house given its features.

This problem can be framed as a **supervised machine learning regression task**. It is supervised because each house comes with both its features and its actual sale price. The model is to be trained to learn the relationship between these features and the sale price, allowing it to make predictions on new, unseen data. Since the target variable, sale price, is a continuous numerica value, the best type of model for this task is a regression model.

The performance of the model is evaluated using the **Root Mean Squared Logarithmic Error (RMSLE)**. This metric is good for housing price prediction because it measures the relative differences between predicted and actual values. It helps reduce the influence of extremely high prices and focuses more on proportional errors.

# 2. Get the Data (15 pts)

Minimum expectations:

1. Load the data.
2. Check the size and type of data.

In [ ]:
# Load the data
training_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv', index_col="Id")
testing_data = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/test.csv',  index_col="Id")


# Check the size and type of data
training_data.info()

After loading the dataset and checking its structure, it can be observed that the data consists of **1,460 rows and 80 columns**. the rows are the house instances with their distinctive ID values, and the columns are the attributes of each house which can be divided into features (inputs) and targets (outputs). The dataset contains a mix of numerical and categorical attributes, with 34 columns represented as **integers**, 3 as **floating-point numbers**, and 43 as **categorical** (object) data types. This combination of data types indicates that both numerical processing and categorical encoding will be necessary during preprocessing. Also, several columns have missing values which will require imputation or removal depending on their importance.

# 3. Explore the Data (15 pts)

General expectations

1. Discuss the types of features (categorical, ordinal, int/float, bounded/unbounded, text, etc.)
2. What features have missing values?
3. Are there any outliers? If so, how can you tell?
4. What features appear usefulness for the task? Are there any that appear useless? Why?
5. Identify and characterize the target distribution.
6. Visualize the data.
7. Study the correlations between attributes.


In [ ]:
%%bash
cat /kaggle/input/house-prices-advanced-regression-techniques/data_description.txt

# Types of Features
The dataset contains a mix of feature types, which can be categorized into categorical, ordinal, integers, floating point numbers,text, and boolean. Categorical features take on a limited set of descriptive values without a speicific order, such as MSZoning, RoofStyle, or Neighborhood, which describe groups or classes. Ordinal features are also descriptive but have a distinct ordering, like OverallQual, ExterQual, or BsmtCond, where higher values indicate better quality or condition. Numeric features are quantitative values that can be measured and used in calculations; these include integers, like GrLivArea or GarageCars, and floats, like LotFrontage or MasVnrArea. Text features are those that contain free-form strings rather than structured categories or numbers. In this dataset, examples include MiscFeature or Fence, where the values are descriptive text like “Shed” or “GdPrv” rather than numeric scores or fixed categories. Finally, boolean or binary features, such as CentralAir, which give each value in positivie and negative description, typically represented by 0 or 1. Understanding these categories helps in selecting appropriate preprocessing and modeling techniques for each type of feature. After going through the data description file for each feature, we can categroize them as shown in table 1.

**Table 1. Summary of features in the dataset, showing the feature names and their respective types.**

| #  | Feature      | Feature Type    |   | #  | Feature      | Feature Type    |   | #  | Feature      | Feature Type    |   | #  | Feature       | Feature Type  |
|----|--------------|-----------------|---|----|--------------|-----------------|---|----|--------------|-----------------|---|----|---------------|---------------|
| 1  | MSSubClass   | Categorical     |   | 21 | RoofStyle    | Categorical     |   | 41 | CentralAir   | Boolean/Binary  |   | 61 | GarageCars    | Numeric (int) |
| 2  | MSZoning     | Categorical     |   | 22 | RoofMatl     | Categorical     |   | 42 | Electrical   | Categorical     |   | 62 | GarageArea    | Numeric (int) |
| 3  | LotFrontage  | Numeric (float) |   | 23 | Exterior1st  | Categorical     |   | 43 | 1stFlrSF     | Numeric (int)   |   | 63 | GarageQual    | Ordinal       |
| 4  | LotArea      | Numeric (int)   |   | 24 | Exterior2nd  | Categorical     |   | 44 | 2ndFlrSF     | Numeric (int)   |   | 64 | GarageCond    | Ordinal       |
| 5  | Street       | Categorical     |   | 25 | MasVnrType   | Categorical     |   | 45 | LowQualFinSF | Numeric (int)   |   | 65 | PavedDrive    | Ordinal       |
| 6  | Alley        | Categorical     |   | 26 | MasVnrArea   | Numeric (float) |   | 46 | GrLivArea    | Numeric (int)   |   | 66 | WoodDeckSF    | Numeric (int) |
| 7  | LotShape     | Ordinal         |   | 27 | ExterQual    | Ordinal         |   | 47 | BsmtFullBath | Numeric (int)   |   | 67 | OpenPorchSF   | Numeric (int) |
| 8  | LandContour  | Ordinal         |   | 28 | ExterCond    | Ordinal         |   | 48 | BsmtHalfBath | Numeric (int)   |   | 68 | EnclosedPorch | Numeric (int) |
| 9  | Utilities    | Categorical     |   | 29 | Foundation   | Categorical     |   | 49 | FullBath     | Numeric (int)   |   | 69 | 3SsnPorch     | Numeric (int) |
| 10 | LotConfig    | Categorical     |   | 30 | BsmtQual     | Ordinal         |   | 50 | HalfBath     | Numeric (int)   |   | 70 | ScreenPorch   | Numeric (int) |
| 11 | LandSlope    | Ordinal         |   | 31 | BsmtCond     | Ordinal         |   | 51 | BedroomAbvGr | Numeric (int)   |   | 71 | PoolArea      | Numeric (int) |
| 12 | Neighborhood | Categorical     |   | 32 | BsmtExposure | Ordinal         |   | 52 | KitchenAbvGr | Numeric (int)   |   | 72 | PoolQC        | Ordinal       |
| 13 | Condition1   | Categorical     |   | 33 | BsmtFinType1 | Ordinal         |   | 53 | KitchenQual  | Ordinal         |   | 73 | Fence         | Ordinal       |
| 14 | Condition2   | Categorical     |   | 34 | BsmtFinSF1   | Numeric (int)   |   | 54 | TotRmsAbvGrd | Numeric (int)   |   | 74 | MiscFeature   | Categorical   |
| 15 | BldgType     | Categorical     |   | 35 | BsmtFinType2 | Ordinal         |   | 55 | Functional   | Ordinal         |   | 75 | MiscVal       | Numeric (int) |
| 16 | HouseStyle   | Categorical     |   | 36 | BsmtFinSF2   | Numeric (int)   |   | 56 | Fireplaces   | Numeric (int)   |   | 76 | MoSold        | Numeric (int) |
| 17 | OverallQual  | Ordinal         |   | 37 | BsmtUnfSF    | Numeric (int)   |   | 57 | FireplaceQu  | Ordinal         |   | 77 | YrSold        | Numeric (int) |
| 18 | OverallCond  | Ordinal         |   | 38 | TotalBsmtSF  | Numeric (int)   |   | 58 | GarageType   | Categorical     |   | 78 | SaleType      | Categorical   |
| 19 | YearBuilt    | Numeric (int)   |   | 39 | Heating      | Categorical     |   | 59 | GarageYrBlt  | Numeric (float) |   | 79 | SaleCondition | Categorical   |


# Missing Values
Several features contain missing values. For example, Alley, PoolQC, Fence, and MiscFeature have many missing entries, while some numerical features like LotFrontage, MasVnrArea, and GarageYrBlt also have gaps. These missing values will be handled either by imputing with median or most frequent values or by removing of features or instances.

Out of curiosity, I had check the data set of one of the features, namely the alley feature, because according to the data description file, NA is a real class which means no alley. Pandas treat strings like NA, NaN, None...etc. as null values, which means unknown value for a particular instance. The following code takes the columns that have null values then by loading a new a dataframe to investigate whether the null value is a real null or a class which happens to be named in null values string format.

In [ ]:
features_with_nulls = training_data.columns[training_data.isnull().any()].tolist()


# Load dataset preserving strings, only keep columns with nulls
train_nulls = pd.read_csv('/kaggle/input/house-prices-advanced-regression-techniques/train.csv',index_col="Id",keep_default_na=False)

# Keep only the columns that had nulls in the first dataset
train_nulls = train_nulls[features_with_nulls]

# Loop through these columns and print unique values
for col in train_nulls.columns:
    unique_vals = train_nulls[col].unique()     # exact values as in CSV
    print(f"Feature: {col}")
    print("Unique values:", unique_vals)
    print("----------------------------------------------------------------")

Upon examining the dataset, several observations regarding missing values and special cases emerge. *LotFrontage* contains actual missing values labeled as NA, indicating unknown lot frontage measurements. The *Alley* feature has no true missing values; what appears as null in pandas is actually a valid class, NA, which signifies the absence of an alley. Similarly, *MasVnrType* contains NA and None values, where None represents no masonry veneer, and NA may indicate missing or unknown data. Since *MasVnrArea* is a numerical value, we assume that NA means unknown values because we do have zero values. Basement-related features such as *BsmtQual*, *BsmtCond*, *BsmtExposure*, *BsmtFinType1*, and *BsmtFinType2* do not have null values, suggesting complete data for basement quality and type. *Electrical* contains some NA values representing missing entries, while *FireplaceQu*, *GarageType*, *GarageFinish*, *GarageQual*, *GarageCond*, *PoolQC*, *Fence*, and *MiscFeature* appear complete, though *MiscFeature* has sparse and vague entries. *GarageYrBlt* may contain missing values, but it is uncertain without further inspection. Overall, the dataset contains a mix of true missing values and categorical NA entries that have meaning, which requires careful handling during preprocessing.

In [ ]:
# This code finds the nullified value features with the number of null value of each
missing_values = training_data.isnull().sum()

missing_values = missing_values[missing_values > 0]

print(missing_values)

From the previuous discussion, we know that fome features in the dataset have actual missing values that need to be addressed before modeling, while others contain NA entries that represent valid categories rather than unknown data. The following list contains the real missing value features with the number of missing value of each.

1. **LotFrontage:** 259 missing values
1. **MasVnrType:** 872 missing values
1. **MasVnrArea:** 8 missing values
1. **Electrical:** 1 missing values


# Target Distribution:


We can use the description method to see the characterostics of the target distribution. Judging from the statitical values and the graph, we can see that SalePrice target of is right-skewed. We can decrease the skewness by applying square root or log functions on the target values to normalize it. 

In [ ]:
import seaborn as sns
from scipy.stats import skew

target = training_data3['SalePrice']

desc_stats = target.describe()
print("Descriptive statistics:\n", desc_stats)

target_skewness = skew(target)
print("\nSkewness of SalePrice:", target_skewness)

plt.figure(figsize=(10,6))
sns.histplot(target, bins=30, kde=True, color='skyblue')
plt.title('Distribution of SalePrice')
plt.xlabel('SalePrice')
plt.ylabel('Frequency')
plt.show()

# Finding Outliners

Outliers in the dataset were identified by analyzing the numeric features using the Z-score method, a widely used statistical approach. The Z-score measures how many standard deviations a particular data point deviates from the mean of its feature.
, where mean and std are the mean and standard deviation of the feature, respectively. Values with an absolute Z-score greater than 3 are considered outliers, as they lie far from the central distribution of the data. The following code calculates the Z-scores for all numeric features, identifies which values exceed this threshold, and lists the features containing outliers along with their corresponding Z-score values. This approach provides a systematic way to detect extreme values that could affect the statistical properties or predictive models applied to the dataset. The list below shows the features with outliners with their corresponding value of Z-score.

In [ ]:
numeric_features = training_data.select_dtypes(include=['int64', 'float64'])

means = numeric_features.mean()
stds = numeric_features.std()

for col in numeric_features.columns:
    z_scores = (numeric_features[col] - means[col]) / stds[col]
    max_z = np.max(np.abs(z_scores))
    if max_z > 3:
        print(f"Attribute: {col}, Max Z-score: {max_z:.2f}")

# Visualizing the Data

In this section, we will visualize the data in the form of histogram and scatter plot. Also, we will find the standard corroletion coeffient for each feature related to the target value.

In [ ]:
import matplotlib.pyplot as plt


plots = training_data.hist(bins=50, figsize=(12, 8))
plt.tight_layout()

In [ ]:
training_data2 = training_data.copy()

# House age at sale
training_data2["HouseAge"] = training_data2["YrSold"] - training_data2["YearBuilt"]

# Age since last remodeling
training_data2["RemodAge"] = training_data2["YrSold"] - training_data2["YearRemodAdd"]

# Total bathrooms
training_data2["TotalBath"] = (
    training_data2["FullBath"] + training_data2["HalfBath"] +
    training_data2["BsmtFullBath"] + training_data2["BsmtHalfBath"]
)

# Bathroom per room ratio
training_data2["BathRatio"] = training_data2["TotRmsAbvGrd"] / training_data2["TotalBath"] 

# Average living area per room
training_data2["LivingAreaPerRoom"] = training_data2["GrLivArea"] / training_data2["TotRmsAbvGrd"]

# Living area to lot area ratio
training_data2["GrLivAreaPerLot"] = training_data2["GrLivArea"] / training_data2["LotArea"]


numeric_data = training_data2.select_dtypes(include=['number'])

# Compute correlation
corr_matrix = numeric_data.corr()

# Get correlation with SalePrice
corr_with_price = corr_matrix["SalePrice"].sort_values(ascending=False)
print(corr_with_price)

Only values of |R|>0.2 will be shown in scatter plots. We can see that GrLivAreaPerLot has value -0.002416, which is close to 0. Because it is clustered, this means we can't interpret the scatter plot using linear means .

In [ ]:
features = ["GrLivArea", "GarageArea", "TotalBsmtSF", "GrLivAreaPerLot","1stFlrSF",
            
]

plot = (
    pd.plotting
      .scatter_matrix(
          training_data2.loc[:, features + ["SalePrice"]],
          figsize=(12, 8),
          diagonal="kde",
      )
)

In [ ]:
features2 = ["LivingAreaPerRoom", "YearBuilt", "YearRemodAdd","GarageYrBlt", "MasVnrArea",
            
            
]

plot2 = (
    pd.plotting
      .scatter_matrix(
          training_data2.loc[:, features2 + ["SalePrice"]],
          figsize=(12, 8),
          diagonal="kde",
      )
)


In [ ]:
features3 = ["BsmtFinSF1", "LotFrontage", "WoodDeckSF","2ndFlrSF", "OpenPorchSF", 
            
            
]

plot3 = (
    pd.plotting
      .scatter_matrix(
          training_data2.loc[:, features3 + ["SalePrice"]],
          figsize=(12, 8),
          diagonal="kde",
      )
)

In [ ]:
features4 = ["LotArea", "BsmtUnfSF", "RemodAge", "HouseAge"
            
]

plot4 = (
    pd.plotting
      .scatter_matrix(
          training_data2.loc[:, features4 + ["SalePrice"]],
          figsize=(12, 8),
          diagonal="kde",
      )
)

# Features Usefulness

1. Useful Features:

* GrLivArea (float, above-ground living area in sq ft) and TotalBsmtSF (float, basement area) are highly predictive due to strong positive correlation with house prices. More usable space directly increases property value.

* LotArea and LotFrontage (float) are moderately useful; they represent the total property size and street exposure. Applying log transformation reduces skewness and improves model performance.

* HouseAge (YrSold - YearBuilt) and RemodAge (YrSold - YearRemodAdd) quantify property depreciation and remodeling history, capturing factors that strongly influence market value.

* FullBath, HalfBath, TotalBath (floats/ints) and BathRatio (TotalBath/TotRmsAbvGrd) capture living comfort and density.

* TotRmsAbvGrd and LivingAreaPerRoom normalize space usage and provide additional predictive power.

* Neighborhood, MSZoning (categorical) encode location-specific variations in property value and zoning restrictions. One-hot encoding preserves this information for the model.

* 1stFlrSF, BsmtFinSF1, WoodDeckSF, OpenPorchSF, MasVnrArea, KitchenAbvGr (numerical) contribute small but consistent predictive signals related to living area distribution and quality of amenities.

1. Useless Features:

* Alley, PoolQC, MiscFeature, Fence (categorical) are mostly missing or rarely present, providing minimal signal and adding noise to the model.

* MoSold (month sold) and YrSold (year sold) have weak standalone correlation with prices. Without interaction terms, they are less predictive of intrinsic property value.

* Features that are captured indirectly by engineered variables (e.g., some raw bathroom counts) may have diminished usefulness when ratios or totals are used instead.

# 4. Prepare the Data for ML (15 pts)

Use the Scikit Learn data pipeline and preprocessing API. Write functions for any custom data transformations used that are not available in the Scikit Learn API.

Minimum expectations. Some of these may or may not be necessary for this task. If a step is not necessary, discuss why.

1. Clean the data.
2. Perform feature selection.
3. Perform feature engineering.
5. Standardize or normalize features.

Your data preparation pipeline does not need to use all of the features, but you must use at least one categorical feature, one ordinal feature, and one numeric feature. These can be raw features, or features that you create yourself.

In [ ]:
from sklearn.impute import SimpleImputer
training_data3 = training_data2.copy()
training_data_final = training_data2.copy() # we will use this dataframe at the end to have combined pipeline.


imputer = SimpleImputer(strategy="median")
training_data_num = training_data3.select_dtypes(include=[np.number])
imputer.fit(training_data_num)
var = imputer.transform(training_data_num)
training_data_transformed = pd.DataFrame(var, columns=training_data_num.columns,index=training_data_num.index)

In [ ]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
housing_cat = training_data3[["GarageQual"]]

# We will change the NA to NG, so that pandas give us a correct interpretation for the No Garage class. 
df = pd.DataFrame(housing_cat)
imputer = SimpleImputer(strategy="constant", fill_value="NG")
var2 = imputer.fit_transform(housing_cat)
housing_cat_transformed = pd.DataFrame(var2, columns=housing_cat.columns, index=housing_cat.index)

categories_order = [['Ex', 'Gd', 'TA', 'Fa', 'Po', 'NG']]
ordinal_encoder = OrdinalEncoder(categories = categories_order)
housing_cat_encoded = ordinal_encoder.fit_transform(housing_cat_transformed)
training_data3["GarageQual"] = housing_cat_encoded


In [ ]:
from sklearn.preprocessing import OneHotEncoder
neighborhood_cat = training_data3[["Neighborhood"]]

cat_encoder = OneHotEncoder()
neighborhood_cat_1hot = cat_encoder.fit_transform(neighborhood_cat)


In [ ]:
low_corr_features = [
    "BedroomAbvGr", "ScreenPorch", "PoolArea", "MoSold", "3SsnPorch",
    "BsmtFinSF2", "BsmtHalfBath", "MiscVal", "LowQualFinSF", "YrSold",
    "OverallCond", "MSSubClass", "EnclosedPorch"
]

training_data3.drop(low_corr_features, axis=1, inplace=True)

In [ ]:
# We will check the skewness of each feature to normlize it using the log.

num_cols = training_data3.select_dtypes(include=[np.number]).columns

for col in num_cols:
    skew_value = training_data3[col].skew()
    if skew_value > 1:
        print(f"{col}: {skew_value:.2f}")

In [ ]:
training_data3["SalePrice_log"] = np.log1p(training_data3["SalePrice"])
print(training_data3["SalePrice_log"].skew())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12,5))

plt.subplot(1, 2, 1)
sns.histplot(training_data3['SalePrice'], kde=True, bins=30)
plt.title('Original SalePrice Distribution')

plt.subplot(1, 2, 2)
sns.histplot(training_data3['SalePrice_log'], kde=True, bins=30, color='orange')
plt.title('Log-Transformed SalePrice Distribution')

plt.show()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

training_data_scaled = scaler.fit_transform(training_data3[num_cols])

training_data_scaled = pd.DataFrame(training_data_scaled, columns=num_cols, index=training_data3.index)


In [ ]:
filtered_housing = training_data3.copy()

numeric_features = filtered_housing.select_dtypes(include=['int64', 'float64']).columns

for feature in numeric_features:
    mean_val = filtered_housing[feature].mean()
    std_val = filtered_housing[feature].std()
    new_rows = []
    
    # Loop through each value in the column
    for value in filtered_housing[feature]:
        z = (value - mean_val) / std_val
        if abs(z) <= 3:
            new_rows.append(value)
    
    filtered_housing = filtered_housing[filtered_housing[feature].isin(new_rows)]

In [ ]:

# Scatter plot before removing outliers
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(training_data3['GrLivArea'], training_data3['SalePrice'], color='blue', alpha=0.5)
plt.title('Before Outlier Removal')
plt.xlabel('GrLivArea')
plt.ylabel('SalePrice')

# Scatter plot after removing outliers
plt.subplot(1, 2, 2)
plt.scatter(filtered_housing['GrLivArea'], filtered_housing['SalePrice'], color='green', alpha=0.5)
plt.title('After Outlier Removal')
plt.xlabel('GrLivArea')
plt.ylabel('SalePrice')

plt.tight_layout()
plt.show()

# Transformation Pipeline:

In this section, we will use the training_data_final dataframe that we defined earlier to implement the knowledge we utilized thus far to prepare the dataframe for the next step, which is model training.

In [ ]:

irrelevant_features = ["BedroomAbvGr", "ScreenPorch", "PoolArea", "MoSold", "3SsnPorch",
    "BsmtFinSF2", "BsmtHalfBath", "MiscVal", "LowQualFinSF", "YrSold",
    "OverallCond", "MSSubClass", "EnclosedPorch"]
housing = training_data_final.drop(columns=irrelevant_features)




skewed_features = ["LotFrontage", "LotArea", "MasVnrArea", "BsmtFinSF1",
                   "TotalBsmtSF", "1stFlrSF", "GrLivArea","KitchenAbvGr",
                   "WoodDeckSF", "OpenPorchSF","BathRatio", "GrLivAreaPerLot"]


for col in skewed_features:
    housing[col] = np.log1p(housing[col])

housing_labels = housing["SalePrice"]
housing = housing.drop(columns=["SalePrice"])



default_num_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

preprocessing = ColumnTransformer([
        ("cat", cat_pipeline, make_column_selector(dtype_include=object))
    ],
    remainder=default_num_pipeline 
)

housing_prepared = preprocessing.fit_transform(housing)


# 5. Explore Different Models (15 pts)

Minimum expectations:

1. Train an unregularized linear model, a linear model with default parameters, and a decision tree.
2. Measure and compare their performance.
3. Analyze the most significant variables for each algorithm.
4. Analyze the types of errors the models make.


In [ ]:
# RMSLE of linear regression model without CV

def rmsle(y_true, y_pred):
    y_pred = np.maximum(0, y_pred)
    return np.sqrt(mean_squared_log_error(y_true, y_pred))

rmsle_scorer = make_scorer(rmsle, greater_is_better=False)


lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_reg.fit(housing, housing_labels)

housing_predictions = lin_reg.predict(housing)


rmsle_train = rmsle(housing_labels, housing_predictions)
print("Training RMSLE:", rmsle_train)

In [ ]:
# RMSLE of linear regression model with CV
lin_rmseCV = -cross_val_score(lin_reg, housing, housing_labels,scoring=rmsle_scorer, cv=10)

print(pd.Series(lin_rmseCV).describe())

In [ ]:
# RMSLE of decision tree regression model without CV

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_reg.fit(housing, housing_labels)


housing_predictions_tree = tree_reg.predict(housing)
rmsle_tree = rmsle(housing_labels, housing_predictions_tree)
print("Training RMSLE (Decision Tree):", rmsle_tree)

# We note the model is overfitting because RMSLE close to 0

In [ ]:
# RMSLE of decision tree regression model with CV

tree_cv = -cross_val_score(tree_reg, housing, housing_labels, scoring=rmsle_scorer, cv=10)
print(pd.Series(tree_cv).describe())


# 6. Fine-tune the Models (15 pts)

1. Fine-tune the hyperparameters for a linear model using RandomizedSearchCV.
2. Fine-tune the hyperparameters for a decision tree using GridSearchCV.


In [ ]:
# RMSLE after the use of grid search fine-tune the hyperparameters for decision tree regression model using GridSearchCV.

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("decision_tree", DecisionTreeRegressor(random_state=42)),
])

param_grid = [
    {
        "decision_tree__max_depth": [5, 10, 15, 20, None],
        "decision_tree__min_samples_split": [2, 5, 10],
        "decision_tree__min_samples_leaf": [1, 2, 5],
    }
]

grid_search = GridSearchCV(full_pipeline,param_grid,cv=10,scoring=rmsle_scorer)

grid_search.fit(housing, housing_labels)

print("Best RMSLE:", -grid_search.best_score_)


In [ ]:
# RMSLE after the use of random search fine-tune the hyperparameters for a linear regression model using RandomizedSearchCV.

full_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("model", LinearRegression())])

param_distributions = {
"model__fit_intercept": [True, False],
    "model__positive": [True, False]
}
rnd_search = RandomizedSearchCV(
    estimator=full_pipeline,
    param_distributions=param_distributions,
    n_iter=1000,
    cv=5,
    scoring=rmsle_scorer,
    random_state=42)

rnd_search.fit(housing, housing_labels)

best_model = rnd_search.best_estimator_

housing_predictions = best_model.predict(housing)
rmsle_value = rmsle(housing_labels, housing_predictions)
print("RMSLE:", rmsle_value)


In [ ]:
final_model = rnd_search.best_estimator_  # includes preprocessing

coefficients = final_model["model"].coef_

feature_names = final_model["preprocessing"].get_feature_names_out()

coefficients = coefficients.round(2)

sorted_features = sorted(zip(coefficients, feature_names), key=lambda x: abs(x[0]), reverse=True)



In [ ]:
final_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv")

X_test = testing_data.copy()

# Engineered features (same as in training)
X_test["HouseAge"] = X_test["YrSold"] - X_test["YearBuilt"]
X_test["RemodAge"] = X_test["YrSold"] - X_test["YearRemodAdd"]
X_test["TotalBath"] = X_test["FullBath"] + 0.5 * X_test["HalfBath"]
X_test["BathRatio"] = X_test["TotalBath"] / X_test["TotRmsAbvGrd"]
X_test["LivingAreaPerRoom"] = X_test["GrLivArea"] / X_test["TotRmsAbvGrd"]
X_test["GrLivAreaPerLot"] = X_test["GrLivArea"] / X_test["LotArea"]

# Apply log transform to skewed features (same as training)
skewed_features = ["LotFrontage", "LotArea", "MasVnrArea", "BsmtFinSF1",
                   "TotalBsmtSF", "1stFlrSF", "GrLivArea","KitchenAbvGr",
                   "WoodDeckSF", "OpenPorchSF","BathRatio", "GrLivAreaPerLot"]

for col in skewed_features:
    if col in X_test.columns:
        X_test[col] = np.log1p(X_test[col])

# Now predict
final_predictions = best_model.predict(X_test)

y_test = final_df["SalePrice"].copy() 


# 7. Summarize your results (15 pts)

1. Provide a 2-3 sentence summary of your solution for a non-technical audience of real estate executives.

2. Provide a technical summary of your solution for the ML team lead at a real estate investment company.

**Non-technical summary:**

A predictive system was built to estimate house prices using historical property data and engineered features such as property age, total bathrooms, and living area per room. Numeric data was adjusted to improve accuracy, and the model was optimized to minimize prediction errors. The resulting system provides reliable home value estimates to support investment and decision-making.

**Technical summary:**

A  regression pipeline was constructed to predict residential property prices using historical data from the housing market. The system incorporates multiple engineered features, including property age (HouseAge), time since last remodeling (RemodAge), total bathrooms (TotalBath), bath-to-room ratios (BathRatio), and living area per room metrics (LivingAreaPerRoom and GrLivAreaPerLot), which capture key aspects of property value. Skewed numeric features were log-transformed to stabilize variance and improve model performance. Categorical features were handled via one-hot encoding, while missing values in both numeric and categorical columns were imputed using median and most frequent strategies, respectively. The numeric features were standardized to ensure balanced scaling across predictors.

The predictive model used was LinearRegression, which was optimized through randomized hyperparameter search over fit_intercept and positive parameters, with RMSLE (Root Mean Squared Logarithmic Error) as the scoring metric to penalize underestimation of high-value properties. The pipeline integrates preprocessing and modeling steps, ensuring consistent transformations on both training and unseen test datasets. Predictions were constrained to be non-negative, preserving realistic pricing outputs. Overall, the system is robust, interpretable, and provides accurate property price estimates while accounting for the skewed distribution of housing features.

# 9. Submit your predictions to the competition

In [ ]:
test_predictions = final_predictions

In [ ]:
submission_df = (
    pd.read_csv(
        "/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",
        index_col="Id"
    ).assign(
        SalePrice=test_predictions
    ).to_csv(
        "submission.csv",
        index=True,
    )
)

In [ ]:
%%bash

cat submission.csv | head -n 5